# Bonus — Where do the typical-section parameters come from?

*A side trip: from a real Boeing 737-800 to the seven numbers you've been using in the notebooks.*

In the main lecture notebooks we populate the aeroelastic typical section with a number of parameters: $c$, $e$, $x_\alpha$, $m$, $I_{EA}$, $k_h$, $k_\theta$. The numbers you saw aren't picked at random; they were chosen to imitate a real narrow-body airliner that you've almost certainly flown on: the **Boeing 737-800**, the workhorse of carriers like Ryanair and Southwest.

This notebook is for you if you've ever paused mid-lecture and thought:

> *Where do these numbers actually come from? Do they have any link with a real aircraft?*

It walks through, step by step, how each parameter is derived from publicly available 737-800 data. Every claim is either hyperlinked to a public source or referenced to a well-known textbook.

> 💡 **A note before we start.** This is a *teaching* exercise, not a flight-certification document. The values we'll arrive are meant to give us a perception of realism, but the real aeroelastic analyses on the 737-800 and similar aircraft are far more elaborate. The point here is not to find 100% accurate values, bu rather to make every number in the lecture notebooks feel like it came from a real machine instead of falling out of the sky.

## The typical section parameters

Here's what we're going to derive. Don't worry if some symbols look unfamiliar: we'll explain each one as we go.

| Symbol | Quantity | Units | Set by |
|---|---|---|---|
| $c$ | Section chord | m | aircraft geometry |
| $b = c/2$ | Section half-chord | m | aircraft geometry |
| $e$ | EA offset from mid-chord (normalised by $b$) | — | wing structural layout |
| $x_\alpha$ | CG offset from EA (normalised by $b$) | — | mass distribution, including fuel |
| $m$ | Mass per unit span | kg/m | total wing mass divided by span |
| $I_{EA}$ | Moment of inertia per unit span about EA | kg·m²/m | $m$ and radius of gyration $r_\alpha$ |
| $k_h$ | Heave stiffness | N/m | targets the first wing-bending frequency |
| $k_\theta$ | Pitch stiffness | N·m/rad | targets the first wing-torsion frequency |

### Sign conventions

- Origin at the **half-chord** (mid-chord of the section).
- $x$-axis along the chord, **positive downstream**.
- $h$ positive **upward**, $\theta$ positive **nose-up**.
- $e < 0$ means the elastic axis sits **forward** of mid-chord, $e > 0$ menas the elastic axis sits **aft** of mid-chord.
- $x_\alpha < 0$ means the centre of mass sits **forward** of the EA, $x_\alpha > 0$ means the centre of mass sits **aft** of the EA.

Length scales are normalised by the **half-chord** $b$; this is the standard convention used in aeroelasticity.

Before we start, run the cell below to import the python libraries we'll need. We'll mainly lean on NumPy for the algebra.

In [1]:
import numpy as np
print("NumPy import complete. Ready to fly.")

NumPy import complete. Ready to fly.


## 1. Meet the airplane

The **Boeing 737-800** is a narrow-body twin-engine airliner that entered service in 1998. As of late 2024 over 3,400 of them are still operational, it's the single most-produced variant of the 737 Next Generation family. See the [Wikipedia overview](https://en.wikipedia.org/wiki/Boeing_737_Next_Generation) for context.

We picked it because:

1. **You've probably flown it** if you've travelled with a low-cost airliner! I myself fly with Ryanair often and have flown on the 737-800 countless times 😁
1. Many of its specifications are **exhaustively documented in the public domain**.
2. It's **representative of the narrow-body (single-aisle) aircraft class** employed for short-haul commercial aviation (together with other aircraft such as those from the Airbus A320 family).
3. **It has a traditional swept wing:** two main spars, a wingbox between them, fuel inside that wingbox. No exotic structural concepts we'd have to apologise for.

Our key reference sources, all freely accessible:

- Chris Brady, [**The Boeing 737 Technical Site**](http://www.b737.org.uk/) — the de-facto online reference for 737 specifications.
- B. van der Zalm, [**aircraftinvestigation.info — Boeing 737-800**](https://aircraftinvestigation.info/airplanes/Boeing_737-800.html)
  — an independent analysis with an unusually detailed structural mass breakdown.
- FAA [**AC 25.335-1A**](https://www.faa.gov/documentLibrary/media/Advisory_Circular/25-335-1a.pdf)
  — the advisory circular on design dive-speed margin.
- The [**kitbagpubs.com 737-800 limitations study guide**](https://www.kitbagpubs.com/Updates/737-800Limitations.pdf)
  — operating-speed limits.

Let's pull out the numbers we need.

### Wing geometry

The first chunk of data is purely geometric. Every value below comes from the
[Boeing 737 Technical Site detailed tech specs](http://www.b737.org.uk/techspecsdetailed.htm).

| Quantity | Value |
|---|---|
| Wingspan (no winglets) | $b_\text{full} = 34.32~\text{m}$ |
| Gross wing area | $S = 124.58~\text{m}^2$ |
| Aspect ratio | $\text{AR} = 9.45$ |
| Taper ratio | $\lambda = 0.159$ |
| Root chord | $c_r = 7.88~\text{m}$ |
| Tip chord | $c_t = 1.25~\text{m}$ |
| Mean aerodynamic chord | MAC $= 3.96~\text{m}$ |
| ¼-chord sweep | $\Lambda_{c/4} = 25.02°$ |
| Average thickness-to-chord | $t/c \approx 10\%$ |


In [2]:
# Boeing 737-800 wing geometry (all from b737.org.uk/techspecsdetailed.htm)
span        = 34.32      # full wingspan (no winglets), m
S_ref       = 124.58     # gross wing area, m^2
c_root      = 7.88       # root chord at fuselage centreline, m
c_tip       = 1.25       # tip chord, m
sweep_qc    = 25.02      # quarter-chord sweep, degrees

# Derived quantities
semispan    = span / 2
AR          = span**2 / S_ref
taper       = c_tip / c_root

print(f"Semi-span:       {semispan:.3f} m")
print(f"Aspect ratio:    {AR:.3f}")
print(f"Taper ratio λ:   {taper:.3f}")

Semi-span:       17.160 m
Aspect ratio:    9.455
Taper ratio λ:   0.159


### Speed envelope

We'll want a couple of speeds in the back of our heads, because flutter analysis is all about beating these comfortably.

| Speed | Value | Source |
|---|---|---|
| Long-range cruise | M 0.785 (≈ 230 KIAS, ≈ 450 KTAS) | [b737.org.uk](http://www.b737.org.uk/techspecsdetailed.htm) |
| Maximum operating $V_{MO}/M_{MO}$ | 340 kt CAS / M 0.82 | [kitbagpubs.com](https://www.kitbagpubs.com/Updates/737-800Limitations.pdf) |
| Design dive speed (lower bound) | $V_D \ge V_{MO}/0.8 \approx 425$ kt EAS | [FAA AC 25.335-1A](https://www.faa.gov/documentLibrary/media/Advisory_Circular/25-335-1a.pdf) |
| Service ceiling | 41,000 ft (≈ 12,500 m) | [b737.org.uk](http://www.b737.org.uk/techspecsdetailed.htm) |

The **dive speed** $V_D$ is the certification ceiling: by [FAR 25.335(b)](https://www.faa.gov/documentLibrary/media/Advisory_Circular/25-335-1a.pdf)
the design cruise speed must be no greater than $0.8\,V_D$, so $V_D \ge V_{MO}/0.8$. By [FAR 25.629](https://www.ecfr.gov/current/title-14/chapter-I/subchapter-C/part-25), the flutter speed must in turn beat $V_D$ by at least 15%. So if your model gives a flutter onset at, say, 250 m/s EAS at sea level, that's only just inside the $1.15\,V_D$ envelope. Useful sanity check to keep in mind.

In [3]:
# Speed envelope
KTS_TO_MPS    = 0.5144

V_cruise_TAS  = 230               # m/s TAS at FL350 (≈ M0.785 × 296 m/s)
V_MO_EAS      = 340 * KTS_TO_MPS  # max operating speed, m/s EAS
M_MO          = 0.82
V_D_min_EAS   = V_MO_EAS / 0.8    # FAR 25.335(b) lower bound, m/s EAS

# Densities (ISA)
rho_SL        = 1.225             # sea level, kg/m^3
rho_FL350     = 0.380             # 35,000 ft, kg/m^3

print(f"V_MO  = {V_MO_EAS:6.1f} m/s EAS  (= 340 kt CAS)")
print(f"V_D   ≥ {V_D_min_EAS:6.1f} m/s EAS  (FAR 25.335 lower bound)")
print(f"Cruise = {V_cruise_TAS:5d} m/s TAS")
print(f"ρ at sea level: {rho_SL:.3f} kg/m³")
print(f"ρ at FL350:     {rho_FL350:.3f} kg/m³")


V_MO  =  174.9 m/s EAS  (= 340 kt CAS)
V_D   ≥  218.6 m/s EAS  (FAR 25.335 lower bound)
Cruise =   230 m/s TAS
ρ at sea level: 1.225 kg/m³
ρ at FL350:     0.380 kg/m³


### Wing mass — structural, attached devices and fuel

The wing isn't just the metal making the load-carrying primary structure: we also need to consider the wing-attached devices like flaps, ailerons, and slats, and the fuel carried inside the wingbox.

| Group | Item | Mass, kg |
|---|---|---|
| **Wing primary structure** | Aluminium skins | 3 788 |
|  | Aluminium spars | 2 976 |
|  | Aluminium ribs | 2 499 |
|  | Empty wing fuel tank shells | 146 |
|  | Vent surge tanks | 12 |
| **Wing-attached devices** | Trailing-edge double-slotted flaps | 650.7 |
|  | Ailerons | 119.6 |
|  | Aluminium honeycomb spoilers | 65.1 |
|  | Leading-edge slats | 255.8 |
|  | Bleed-air thermal LE anti-icing | 37.8|
|  | Hydraulic servo actuators | 28.3 |
| **Fuel** | Left tank | 3900|
|  | Right tank | 3900 |

Sources:
- [aircraftinvestigation.info breakdown](https://aircraftinvestigation.info/airplanes/Boeing_737-800.html)
- [Boeing 737 fuel system page](http://www.b737.org.uk/fuel.htm)

In [4]:
# Wing primary structure and attached devices
m_skins = 3788
m_spars = 2976
m_ribs = 2499
m_fuel_tank_shells = 146
m_vent_surge_tanks = 12
m_te_flaps = 650.7
m_ailerons = 119.6
m_spoilers = 65.1
m_le_slats = 255.8
m_le_anti_icing = 37.8
m_actuators = 28.3

# Max fuel mass
m_fuel_max = 3900 * 2

# Print summations
m_wing_struct_total = (
    m_skins
    + m_spars
    + m_ribs
    + m_fuel_tank_shells
    + m_vent_surge_tanks
    + m_te_flaps
    + m_ailerons
    + m_spoilers
    + m_le_slats
    + m_le_anti_icing
    + m_actuators
)
print(f"Total wing structural mass with attached devices: {m_wing_struct_total:.0f} kg")
print(f"Max fuel mass in wing: {m_fuel_max:.0f} kg")

Total wing structural mass with attached devices: 10578 kg
Max fuel mass in wing: 7800 kg


## 2. Picking the typical-section station

Here's the first awkward truth about the typical section: it's a **2-D** model of a **3-D** wing. Real wings change chord, twist, mass, and stiffness as you move from root to tip. To collapse all that variation into *one* section, we have to commit to a specific spanwise location.

Which one?

The classical answer, used in Bisplinghoff, Ashley & Halfman's foundational text *Aeroelasticity* ([Dover link](https://store.doverpublications.com/products/9780486691893?srsltid=AfmBOooOZkg27RmFgh-t6XJUDfYKWFkDlcHoTOWaH405sazKbNkurIQt)), is the **3/4-semi-span station**.

So we evaluate everything at $\eta = y/(b/2) = 0.75$.

### What is the chord there?

We'll assume **linear taper** between the root chord ($c_r = 7.88~\text{m}$) and the tip chord ($c_t = 1.25~\text{m}$). This isn't quite right, as the real 737 wing has a *kink* inboard (the "yehudi break"), where the trailing edge breaks aft to extend the inboard chord. But the kink is well inboard of where we're evaluating, so for the 3/4-semi-span station our linear taper is a good enough approximation. We thus perform a linear interpolation between the value of the root chord at the spanwise station $\eta=0$ and the value of the tip chord at the spanwise station $\eta=1$, and we evaluate it at $\eta=0.75$.

In [5]:
# The reference spanwise station of the typical section
eta_ref = 0.75

# Interpolate between root chord and tip chord
c = np.interp(eta_ref, [0, 1], [c_root, c_tip])
b = c / 2

# Print results
print(f"Chord at η = {eta_ref}:  c = {c:.1f} m")
print(f"Half-chord:         b = {b:.2f} m")


Chord at η = 0.75:  c = 2.9 m
Half-chord:         b = 1.45 m


## 3. Chord $c$ and half-chord $b$

That's it for the first two parameters. We just read them off the planform at the chosen station:

$$
\boxed{\; c = 2.9~\text{m}, \quad b = c/2 = 1.45~\text{m} \;}
$$


## 4. Elastic-axis offset $e$

### The physical picture

The **elastic axis (EA)** of a wing is the spanwise locus of **shear centres** of successive cross-sections. The shear centre of a cross-section is the point through which a transverse load must pass to produce bending with no twist; equivalently, a pure torque applied about the shear centre produces twist with no bending. In other words, the EA is the pivot about which torsional deformation occurs.

For a thin-walled multi-cell box section (which is what a real wing is), computing the shear centre exactly requires solving a statically indeterminate shear-flow problem across all cells (see Megson, *Aircraft Structures for Engineering Students*, §23.5 ([Elsevier link](https://www.sciencedirect.com/book/9780080969053/aircraft-structures-for-engineering-students))). As a working approximation we assume that all direct (bending) stress is carried by the four spar-cap booms, that the skin panels carry shear only, and that the wingbox is doubly-symmetric (equal front and rear spar caps, symmetric top and bottom). In this idealization the bending centroid coincides with the geometric centroid of the boom areas, and for a doubly-symmetric section the shear centre coincides with the bending centroid. The EA therefore reduces to the **centroid of the spar caps**.

### Where are the spars?

Boeing doesn't publish the exact spar positions for the 737. The standard textbook reference for jet-transport wing layouts is Egbert Torenbeek's *Synthesis of Subsonic Airplane Design* (1982, [Springer Nature link](https://link.springer.com/book/10.1007/978-94-017-3202-4)), which for typical two-spar transport wings indicates:

- **Front spar at ~15% of chord**
- **Rear spar at ~60% of chord**

### Computing $e$

Placing the EA at the centroid of the two spar positions gives:

$$
\frac{x_{EA}}{c} = \frac{0.15 + 0.60}{2} = 37.5\%
$$

The parameter $e$ is the distance from the mid-chord point nondimensionalized by the half-chord:

$$
e = 2\!\left(\frac{x_\mathrm{EA}}{c} - 0.5\right) = 2\,(0.375 - 0.50) = \boxed{-0.25}
$$


In [6]:
# Spar positions (Torenbeek 1982 heuristic for jet transports)
x_front_spar_frac = 0.15
x_rear_spar_frac  = 0.60
x_spar_centroid   = (x_front_spar_frac + x_rear_spar_frac) / 2   # = 0.375

# Calculate elastic axis offset nondimensionalized by the half-chord
e = 2 * (x_spar_centroid - 0.5)

print(f"Spar-cap centroid:  {x_spar_centroid*100:.1f}% of chord")
print(f"  → e = {e:+.2f}")

Spar-cap centroid:  37.5% of chord
  → e = -0.25


We can consider this value good enough for our notebooks, but it's good to keep in mind how the simplifications we made separate our estimate from the true elastic-axis position in a real wing. The idealized wingbox model assumes all direct stress is carried by four equal spar-cap booms while the skin is purely a shear panel, whereas in reality the skin contributes a distributed fraction of the bending stiffness and the spar caps are not equal in area. The cross-section is treated as doubly symmetric, while the actual airfoil is cambered, the upper and lower skins have different wall thickness, as well as the front and rear spar have different wall thickness and height. A single representative section is used for the entire span, whereas skins and spars properties vary continuously from root to tip, so the elastic axis is in reality a curved spatial line rather than a straight one. Finally, the spar positions themselves are taken from a class-level reference rather than 737-specific data, introducing an additional layer of geometric uncertainty.

## 5. CG offset $x_\alpha$

This is where it gets interesting. As we've seen earlier, the wing carries many masses beyond its own structural mass, and each sits at a different spanwise and chordwise location. In addition, the fuel mass is not constant, as it decreases during flight. These considerations are hopefully an evident hint that developing and maintaining a mass model of an aircraft is a very challenging task.

By making again the idealization of a doubly symmetric wingbox for our typical section, we might be tempted to place the CG at 37.5% of the chord, in the same position of the EA. However, the trailing edge devices are heavier than the leading edge devices, so it's reasonable to expect that the CG is pulled further aft. With this reasoning in mind, we place the CG at 45% of the chord, and we can calculate the nondimensional offset $x_\alpha$ as:

$$
x_\alpha = 2\!\left(\frac{x_\mathrm{CG}}{c} - \frac{x_\mathrm{EA}}{c}\right) = 2\,(0.45 - 0.375) = \boxed{0.15}
$$

In [7]:
x_cg = 0.45
x_alpha = 2 * (x_cg - x_spar_centroid)
print(f"x_alpha = {x_alpha:.2f}")

x_alpha = 0.15


## 6 · Sectional mass $m$

Now we need to express the wing's mass as a **mass per unit span** for the typical section.

### Uniform-strip assumption

The simplest thing is to assume the mass is **uniformly distributed along the span**. This is a coarse approximation, as real wings have more mass near the root and less at the tip, but it follows the spirit of finding ballpark estimates rather than 100% accurate values. For the fuel mass in the wingbox, we assume a configuration with half-filled tanks.

The mass of the typical section is thus obtained by summing the masses in the wing and dividing by the span.

In [8]:
# Calculate total wing mass (structural + attached devices + half of the max fuel mass)
m_total = m_wing_struct_total + m_fuel_max*0.5

# Calculate sectional mass
m = m_total/span
print(f"Typical section mass: {m:.0f} kg/m")

Typical section mass: 422 kg/m


So we'll use:

$$
\boxed{\; m \;=\; 422~\text{kg/m} \;}.
$$


## 7 · Sectional moment of inertia $I_{EA}$

Rather than measure $I_\mathrm{EA}$ directly (very hard as we would need a detailed mass distribution chordwise), we can express it through the **dimensionless radius of gyration** $r_\alpha$:

$$
I_\mathrm{EA} \;=\; m \,(r_\alpha\, b)^2, \qquad r_\alpha \equiv \frac{1}{b}\sqrt{\frac{I_\mathrm{EA}}{m}}
$$

$r_\alpha$ indicates how the mass is distributed relative to an axis, and it essentially represents the distance from that axis where the entire mass could be concentrated to achieve the same moment of inertia.

Although a typical value of $r_\alpha$ for aluminium transport wings is difficult to find, a value commonly employed in worked examples in foundational aeroelastic texts is 0.5 (see Bisplinghoff, Ashley & Halfman's *Aeroelasticity*, or Hodge & Pierce's *Introduction to Structural Dynamics and Aeroelasticity*, [Cambridge link](https://www.cambridge.org/core/books/introduction-to-structural-dynamics-and-aeroelasticity/38F8A0FF59844464F5034E8E0EA8BD31)), so we calculate $I_\mathrm{EA}$ based on this number and on the values of the sectional mass $m$ and of the semi-chord $b$.

In [9]:
# Radius of gyration
r_alpha = 0.5

# Use the sectional mass and the half-chord we already computed
I_EA = m * (r_alpha * b)**2
print(f"I_EA = {I_EA:.0f} kg·m²/m")


I_EA = 223 kg·m²/m


So:

$$
\boxed{\; I_\mathrm{EA} \;=\; 223~\text{kg}\cdot\text{m}^2/\text{m} \;}
$$


## 8. Heave stiffness $k_h$ and pitch stiffness $k_\theta$

The strategy to find $k_h$ and $k_\theta$ is to target a desired natural frequency for the isolated heave and pitch harmonic oscillators:

$$
\omega_h \;=\; \sqrt{\frac{k_h}{m}} \quad \rightarrow \quad k_h \;=\; m \,\omega_h^2 \;=\; m \,(2\pi f_h)^2
$$

$$
\omega_\theta = \sqrt{\frac{k_\theta}{I_\mathrm{EA}}} \quad \rightarrow \quad k_\theta = I_\mathrm{EA}\omega_\theta^2 = I_\mathrm{EA}\,(2\pi f_\theta)^2
$$

### What are reasonable $f_h$ and $f_\theta$?

Even though our section is a 2-D strip, we can think of $f_h$ and $f_\theta$ as a proxy for the first **wing-bending** and wing-torsion mode of the whole 3-D wing.

Following some worked examples from Wright & Cooper's *Introduction to Aircraft Aeroelasticity and Loads* ([Wiley link](https://onlinelibrary.wiley.com/doi/book/10.1002/9781118700440)), we'll target $f_h = 3~\text{Hz}$ and $f_\theta = 9~\text{Hz}$ as representative values.

In [10]:
# Target frequencies, Hz
f_h = 3.0
f_theta = 9.0

# Calculate heave stiffness
omega_h = 2 * np.pi * f_h
k_h = m * omega_h**2

# Calculate pitch stiffness
omega_theta = 2 * np.pi * f_theta
k_theta = I_EA * omega_theta**2

# Print results
print(f"Target f_h = {f_h} Hz → ω_h = {omega_h:.2f} rad/s")
print(f"k_h = {k_h:.1e} N/m\n")
print(f"Target f_theta = {f_theta} Hz → ω_theta = {omega_theta:.2f} rad/s")
print(f"k_theta = {k_theta:.1e} Nm/rad")

Target f_h = 3.0 Hz → ω_h = 18.85 rad/s
k_h = 1.5e+05 N/m

Target f_theta = 9.0 Hz → ω_theta = 56.55 rad/s
k_theta = 7.1e+05 Nm/rad


So:

$$
\boxed{\; k_h \approx 1.5\times 10^5~\text{N/m} \;}
$$

$$
\boxed{\; k_\theta \approx 7.1\times 10^5~\text{Nm/rad} \;}
$$

## 10. Putting it all together

Here are all the parameters in one place, rounded to the values you'll find in the lecture notebooks:

| Symbol | Value | What it captures |
|---|---|---|
| $c$ | 2.9 m | chord at 75% semi-span |
| $b$ | 1.45 m | half-chord |
| $e$ | −0.25 | EA at 37.5% c (center of guessed wingbox position) |
| $x_\alpha$ | +0.15 | CG aft of EA, with half fuel |
| $m$ | 422 kg/m | wing + fuel, uniform strip |
| $I_{EA}$ | 223 kg·m²/m | with $r_\alpha = 0.5$ |
| $k_h$ | $1.5\cdot 10^5$ N/m | targets $f_h = 3$ Hz |
| $k_\theta$ | $7.1\cdot 10^5$ N·m/rad/m | targets $f_\theta = 9$ Hz |

Looking back over the notebook, every step of the derivation required at least one simplifying assumption. We assumed linear taper to compute the chord at the 3/4 semi-span station, even though the real wing has an inboard kink. We placed the elastic axis at the centroid of two spar positions taken from a class-level textbook reference rather than from actual Boeing structural drawings. We estimated the section CG from a qualitative argument about the relative mass of trailing-edge and leading-edge devices rather than from a full spanwise integration of the mass breakdown. We spread the entire wing mass uniformly along the span, when in reality the root section is significantly heavier than the outboard section, and we left out two of the heaviest concentrated masses on the wing entirely: the engine pylon and the winglet. We chose $r_\alpha = 0.5$ because it appears repeatedly in textbook worked examples, not because it was measured for the 737-800. And similarly we calibrated $k_h$ and $k_\theta$ against first wing-bending and torsion frequencies drawn from textbook worked examples, since of course Boeing ground vibration test data is not available.

Each of these choices is individually defensible, as the alternatives would have required either structural data that is not publicly available or a level of modelling complexity disproportionate to the purpose of this exercise, but they do accumulate. Thus we should not be surprised if one day we will find out that the derived parameters deviate substantially from those that can be obtained from a detailed finite-element mass and stiffness model. For the purposes of the lecture notebooks this is entirely adequate: the goal is not to reproduce the actual flutter speed of the 737-800, but to give ourselves parameters grounded enough in a real machine so that the mechanisms we study feel like they connect to something tangible rather than floating in the abstract.

---

If you spot a mistake or have a suggestion, [open an issue on the course repo](https://github.com/fmamitrotta/aeroelastic-instabilities-jupyter) or send me an email: [francesco.mitrotta@uc3m.es](mailto:francesco.mitrotta@uc3m.es).

Happy learning! 🛬